In [1]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [2]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

In [3]:
from __future__ import annotations


def opendoor_stats_v2_exporter(
    input_path: str,
    *,
    output_onefile_jsonl: str = "OPENDOOR/onefile.jsonl",
    output_summary_csv: str = "OPENDOOR/summary.csv",
    output_best_params_jsonl: str = "OPENDOOR/best_params.jsonl",
    # entry: nearest point to entry_hm (searched from BOTH sides) inside [entry_window_from, entry_window_to]
    entry_hm: tuple = (9, 20),
    entry_window_from: tuple = (9, 0),
    entry_window_to: tuple = (9, 25),
    # exit classes: Stack% move is always measured against Stack%_entry
    exit_hm: dict = None,   # {"10m": (9, 40), "30m": (10, 0)}
    # bins (signed — negative and positive values are separate bins)
    stack_bin_min: float = -15.0,
    stack_bin_max: float = 15.0,
    stack_bin_step: float = 1.0,
    bench_bin_min: float = -15.0,
    bench_bin_max: float = 15.0,
    bench_bin_step: float = 1.0,
    devsig_bin_min: float = -5.0,
    devsig_bin_max: float = 5.0,
    devsig_bin_step: float = 0.3,
    # best params selection
    best_min_rate: float = 0.60,
    best_min_total: int = 10,
    # global filter
    min_events_per_ticker: int = 10,
    # column names (all three are per-row, time-varying snapshot values)
    STOCK_NUM_FIELD: str = "Stack%",
    BENCH_NUM_FIELD: str = "Bench%",
    DEVSIG_FIELD: str = "DevSig",
    # ADVANCED: entry is still anchored at entry_hm (9:20) — ADVANCED only pools EXTRA
    # historical (entry,exit) observations from every hourly checkpoint of the day
    # (H:00 -> H:10 / H:00 -> H:30) into a SEPARATE, much larger bin set, and picks its
    # own best_params from that pooled dataset. The "standard" 9:20-only best_params
    # is always computed too and is never replaced by ADVANCED.
    #
    # advanced_offset_minutes is INTENTIONALLY separate from exit_hm: exit_hm's "10m"/"30m"
    # keys are historically named for minutes-after-MARKET-OPEN (09:30), so 9:40/10:00 are
    # actually +20/+40 minutes after entry_hm (09:20) — NOT +10/+30. Reusing that entry-to-exit
    # gap for arbitrary hourly checkpoints would silently misalign the ADVANCED exits. Instead
    # each class explicitly maps to "minutes after its own H:00 checkpoint" here.
    enable_advanced: bool = True,
    advanced_offset_minutes: dict = None,  # {"10m": 10, "30m": 30} — minutes after each H:00
    advanced_hours: Optional[List[int]] = None,  # None = every hour that has data
    # reading
    assume_sorted: bool = True,
    parquet_use_pyarrow: bool = True,
    csv_chunksize: int = 500_000,
    log_every_n_chunks: int = 5,
):
    """
    OpenDoor v2:

    ENTRY (per ticker, per day):
      - Window [entry_window_from, entry_window_to] (default 09:00-09:25).
      - Pick the row whose timestamp is CLOSEST to entry_hm (default 09:20), searching
        both before and after 09:20 within the window (not just "last value <= 09:20").
      - Capture 3 factors from that single snapshot: Stack%, DevSig, Bench%.

    EXIT (per ticker, per day):
      - Two classes: "10m" (09:40) and "30m" (10:00) by default.
      - move = Stack%_exit - Stack%_entry  ->  "long" if move > 0 else "short".
        (move is always measured on Stack%; DevSig/Bench% are entry-side predictors only.)

    RATING per (parameter in {stack, devsig, bench}) x (class in {10m, 30m}) x (bin):
      - total, up, down
      - long_rate = up/total, short_rate = down/total, long_short_ratio = up/down (if down>0)
      - avg_long_move  = mean(move | move>0 in this bin)
      - avg_short_move = mean(move | move<0 in this bin)

    Bins are signed floor-bins: stack/bench step=1.0, devsig step=0.3 (configurable).

    best_params: per parameter x class x direction, bins with rate>=best_min_rate and
    total>=best_min_total are stitched into consecutive intervals (same rule as v1),
    scored by rate*log1p(total), carrying weighted avg_long_move/avg_short_move through
    the merge.

    ADVANCED (enable_advanced=True): in addition to the standard 9:20-anchored dataset,
    every hourly checkpoint (H:00, for whichever hours have data that day) is treated as
    an extra entry point, with exits at H:10 and H:30 — mirroring the same "10m"/"30m"
    classes but sampled many more times per day. This pooled, much larger dataset feeds
    a SEPARATE "advanced" bin set and best_params selection. The real, applied signal
    still always anchors at entry_hm (09:20) — advanced only widens the historical
    sample used to *rate* each bin/parameter, it does not change where the ticker
    actually gets evaluated for trading.
    """
    import gc, json, time, math, gzip
    from collections import defaultdict
    from datetime import datetime
    from typing import Optional, List
    import numpy as np
    import pandas as pd
    from pathlib import Path

    if exit_hm is None:
        exit_hm = {"10m": (9, 40), "30m": (10, 0)}
    if advanced_offset_minutes is None:
        advanced_offset_minutes = {"10m": 10, "30m": 30}

    CLASSES = list(exit_hm.keys())
    PARAMS = ("stack", "devsig", "bench")

    entry_hm_min = entry_window_from[0] * 60 + entry_window_from[1]
    entry_hm_max = entry_window_to[0] * 60 + entry_window_to[1]
    entry_hm_target = entry_hm[0] * 60 + entry_hm[1]

    Path(output_onefile_jsonl).parent.mkdir(parents=True, exist_ok=True)
    Path(output_summary_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(output_best_params_jsonl).parent.mkdir(parents=True, exist_ok=True)

    def _open_gz(path, mode="wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    # lo/hi are the winning interval's bin boundaries — included so a live consumer can check
    # "does the ticker's CURRENT value fall in this good range" from summary.csv alone, without
    # a per-ticker onefile.jsonl fetch. avg_move is the winning interval's avg_long_move/
    # avg_short_move, for a live MINMOVE threshold check.
    BEST_FIELDS = ("rate", "total", "lo", "hi", "avg_move")
    # "adv_" columns mirror the standard ones exactly but are sourced from the hourly-pooled
    # ADVANCED bin set (best_params.advanced) instead of the 09:20-only standard set — only
    # present when enable_advanced=True.
    summary_cols = (
        ["ticker", "bench", "events_total", "days_with_entry", "advanced_events_total"] +
        [f"{p}_{c}_best_{d}_{f}" for p in PARAMS for c in CLASSES for d in ("long", "short") for f in BEST_FIELDS] +
        [f"adv_{p}_{c}_best_{d}_{f}" for p in PARAMS for c in CLASSES for d in ("long", "short") for f in BEST_FIELDS] +
        ["corr", "beta"]
    )
    pd.DataFrame(columns=summary_cols).to_csv(output_summary_csv, index=False, mode="w")

    onefile_f     = _open_gz(output_onefile_jsonl, "wt")
    best_params_f = _open_gz(output_best_params_jsonl, "wt")
    best_params_f.write(json.dumps({
        "meta": {"version": "opendoor_v2", "generated_at": datetime.utcnow().isoformat() + "Z"}
    }) + "\n")

    # ── helpers ──────────────────────────────────────────────────────────────
    def _js(x):
        if x is None: return None
        if isinstance(x, (np.floating, float)):
            return None if (np.isnan(x) or np.isinf(x)) else round(float(x), 6)
        if isinstance(x, (np.integer, int)): return int(x)
        if isinstance(x, (np.bool_, bool)): return bool(x)
        return x

    def _ok(x):
        try: return np.isfinite(float(x))
        except Exception: return False

    def _clamp(v, lo, hi): return max(lo, min(hi, v))

    def _sbin(v, lo, hi, step):
        # Signed floor-bin: value v falls into [floor(v/step)*step, +step). Negative and
        # positive values land in distinct bins on either side of 0.0 — no special-casing
        # needed since math.floor already handles the sign correctly.
        if not _ok(v): return None
        v = _clamp(float(v), lo, hi)
        return f"{round(math.floor(v / step) * step, 6):.1f}"

    def stack_bin(v):  return _sbin(v, stack_bin_min,  stack_bin_max,  stack_bin_step)
    def bench_bin(v):  return _sbin(v, bench_bin_min,  bench_bin_max,  bench_bin_step)
    def devsig_bin(v): return _sbin(v, devsig_bin_min, devsig_bin_max, devsig_bin_step)

    BIN_FN   = {"stack": stack_bin, "devsig": devsig_bin, "bench": bench_bin}
    BIN_STEP = {"stack": stack_bin_step, "devsig": devsig_bin_step, "bench": bench_bin_step}

    def _score(rate, total): return float(rate) * math.log1p(int(total))

    def _new_bin_stat():
        return {"long": 0, "short": 0, "total": 0, "long_sum": 0.0, "short_sum": 0.0}

    def _new_bin_store():
        # bin_store[param][cls][bin_label] -> stat dict
        return {p: {c: defaultdict(_new_bin_stat) for c in CLASSES} for p in PARAMS}

    def _accumulate_class(bin_store, cls, entry_vals, move):
        direction = "long" if move > 0 else "short"
        for p in PARAMS:
            b = BIN_FN[p](entry_vals.get(p))
            if b is None:
                continue
            st = bin_store[p][cls][b]
            st["total"] += 1
            st[direction] += 1
            if direction == "long":
                st["long_sum"] += move
            else:
                st["short_sum"] += move

    # ── per-ticker state ──────────────────────────────────────────────────────
    cur_ticker = None
    cur_day    = None
    bench_seen = None
    static_set = False
    corr_s = beta_s = None

    # standard (09:20-anchored) daily accumulators
    day_entry = None        # {"stack":..,"devsig":..,"bench":..}
    day_entry_dist = None   # |minutes - entry_hm_target| of the currently-held candidate
    day_exits = {}          # cls -> Stack%_exit
    day_count = 0

    # advanced (hourly-pooled) daily accumulators
    day_hour_entry = {}     # hour -> {"stack":..,"devsig":..,"bench":..}
    day_hour_exits = {}     # hour -> {cls -> Stack%_exit}

    bins_std = _new_bin_store()
    bins_adv = _new_bin_store() if enable_advanced else None
    adv_events_total = 0

    def _reset_ticker():
        nonlocal bench_seen, static_set, corr_s, beta_s
        nonlocal day_entry, day_entry_dist, day_exits, day_count
        nonlocal day_hour_entry, day_hour_exits, bins_std, bins_adv, adv_events_total
        bench_seen = None; static_set = False; corr_s = beta_s = None
        day_entry = None; day_entry_dist = None; day_exits = {}; day_count = 0
        day_hour_entry = {}; day_hour_exits = {}
        bins_std = _new_bin_store()
        bins_adv = _new_bin_store() if enable_advanced else None
        adv_events_total = 0

    def _reset_day():
        nonlocal day_entry, day_entry_dist, day_exits, day_hour_entry, day_hour_exits
        day_entry = None; day_entry_dist = None; day_exits = {}
        day_hour_entry = {}; day_hour_exits = {}

    def _finalize_day():
        nonlocal day_count, adv_events_total
        if day_entry is not None:
            stack_e = day_entry["stack"]
            day_count += 1
            for c in CLASSES:
                exit_stack = day_exits.get(c)
                if exit_stack is None or not _ok(exit_stack):
                    continue
                move = float(exit_stack) - float(stack_e)
                _accumulate_class(bins_std, c, day_entry, move)

        if enable_advanced:
            for h, ev in day_hour_entry.items():
                if advanced_hours is not None and h not in advanced_hours:
                    continue
                stack_e = ev["stack"]
                exits_h = day_hour_exits.get(h, {})
                hit = False
                for c in CLASSES:
                    exit_stack = exits_h.get(c)
                    if exit_stack is None or not _ok(exit_stack):
                        continue
                    move = float(exit_stack) - float(stack_e)
                    _accumulate_class(bins_adv, c, ev, move)
                    hit = True
                if hit:
                    adv_events_total += 1

    def _rating(st):
        tot = int(st["total"]); long_cnt = int(st["long"]); short_cnt = int(st["short"])
        return {
            "total": tot, "long": long_cnt, "short": short_cnt,
            "long_rate": round(long_cnt / tot, 4) if tot else None,
            "short_rate": round(short_cnt / tot, 4) if tot else None,
            "long_short_ratio": round(long_cnt / short_cnt, 4) if short_cnt > 0 else None,
            "avg_long_move": round(st["long_sum"] / long_cnt, 4) if long_cnt else None,
            "avg_short_move": round(st["short_sum"] / short_cnt, 4) if short_cnt else None,
        }

    def _best_for_param_class(bins_d, direction, step):
        # Same consecutive-bin stitching as v1's _best_1d, generalized to also carry
        # weighted avg_long_move/avg_short_move (via long_sum/short_sum) through the merge.
        eligible = []
        for b_str, st in bins_d.items():
            tot = int(st["total"])
            if tot < best_min_total: continue
            cnt = int(st[direction])
            rate = cnt / tot if tot else 0.0
            if rate >= best_min_rate:
                try: eligible.append((float(b_str), b_str, dict(st)))
                except ValueError: pass
        eligible.sort(key=lambda x: x[0])
        if not eligible: return []

        intervals = []
        lo_s, hi_f, hi_s = eligible[0][1], eligible[0][0], eligible[0][1]
        agg = dict(eligible[0][2])

        for v, s, st in eligible[1:]:
            if abs(v - (hi_f + step)) < 1e-9:
                hi_f, hi_s = v, s
                for k in ("long", "short", "total", "long_sum", "short_sum"):
                    agg[k] += st[k]
            else:
                intervals.append((lo_s, hi_s, agg))
                lo_s, hi_f, hi_s = s, v, s
                agg = dict(st)
        intervals.append((lo_s, hi_s, agg))

        result = []
        for lo_s, hi_s, agg in intervals:
            r = _rating(agg)
            tot, cnt = r["total"], r[direction]
            rate = cnt / tot if tot else 0.0
            if tot >= best_min_total and rate >= best_min_rate:
                result.append({
                    "lo": lo_s, "hi": hi_s, "total": tot,
                    direction: cnt, "rate": round(rate, 4),
                    "avg_long_move": r["avg_long_move"], "avg_short_move": r["avg_short_move"],
                    "score": round(_score(rate, tot), 4),
                })
        result.sort(key=lambda x: x["score"], reverse=True)
        return result

    def _best_params_block(bin_store):
        best = {}
        for p in PARAMS:
            best[p] = {}
            for c in CLASSES:
                best[p][c] = {
                    "long":   _best_for_param_class(bin_store[p][c], "long",   BIN_STEP[p]),
                    "short": _best_for_param_class(bin_store[p][c], "short", BIN_STEP[p]),
                }
        return best

    # ── flush ticker ──────────────────────────────────────────────────────────
    def _flush():
        if cur_ticker is None:
            return
        _finalize_day()

        events_total = max(
            (int(sum(st["total"] for st in bins_std[p][c].values())) for p in PARAMS for c in CLASSES),
            default=0,
        )
        if events_total < min_events_per_ticker:
            _reset_ticker()
            return

        ratings_std = {p: {c: {b: _rating(st) for b, st in bins_std[p][c].items()} for c in CLASSES} for p in PARAMS}
        best_std = _best_params_block(bins_std)

        ratings_adv = None
        best_adv = None
        if enable_advanced:
            ratings_adv = {p: {c: {b: _rating(st) for b, st in bins_adv[p][c].items()} for c in CLASSES} for p in PARAMS}
            best_adv = _best_params_block(bins_adv)

        payload = {
            "ticker": cur_ticker,
            "bench": bench_seen,
            "static": {"corr": _js(corr_s), "beta": _js(beta_s)},
            "events_total": int(events_total),
            "days_with_entry": int(day_count),
            "advanced_events_total": int(adv_events_total) if enable_advanced else 0,
            "params": {
                "entry_hm": list(entry_hm),
                "entry_window": [list(entry_window_from), list(entry_window_to)],
                "exit_hm": {c: list(t) for c, t in exit_hm.items()},
                "stack_bins":  {"min": stack_bin_min,  "max": stack_bin_max,  "step": stack_bin_step},
                "bench_bins":  {"min": bench_bin_min,  "max": bench_bin_max,  "step": bench_bin_step},
                "devsig_bins": {"min": devsig_bin_min, "max": devsig_bin_max, "step": devsig_bin_step},
                "best_min_rate": best_min_rate,
                "best_min_total": best_min_total,
                "enable_advanced": enable_advanced,
                "advanced_offset_minutes": advanced_offset_minutes if enable_advanced else None,
            },
            "standard": {"ratings": ratings_std},
            "best_params": {"standard": best_std},
        }
        if enable_advanced:
            payload["advanced"] = {"ratings": ratings_adv}
            payload["best_params"]["advanced"] = best_adv

        onefile_f.write(json.dumps(payload, ensure_ascii=False) + "\n")

        row = {
            "ticker": cur_ticker, "bench": bench_seen,
            "events_total": int(events_total), "days_with_entry": int(day_count),
            "advanced_events_total": int(adv_events_total) if enable_advanced else 0,
        }
        for p in PARAMS:
            for c in CLASSES:
                long_best = best_std[p][c]["long"][0] if best_std[p][c]["long"] else None
                short_best = best_std[p][c]["short"][0] if best_std[p][c]["short"] else None
                row[f"{p}_{c}_best_long_rate"]     = _js(long_best["rate"]) if long_best else None
                row[f"{p}_{c}_best_long_total"]    = int(long_best["total"]) if long_best else None
                row[f"{p}_{c}_best_long_lo"]       = long_best["lo"] if long_best else None
                row[f"{p}_{c}_best_long_hi"]       = long_best["hi"] if long_best else None
                row[f"{p}_{c}_best_long_avg_move"] = _js(long_best["avg_long_move"]) if long_best else None
                row[f"{p}_{c}_best_short_rate"]     = _js(short_best["rate"]) if short_best else None
                row[f"{p}_{c}_best_short_total"]    = int(short_best["total"]) if short_best else None
                row[f"{p}_{c}_best_short_lo"]       = short_best["lo"] if short_best else None
                row[f"{p}_{c}_best_short_hi"]       = short_best["hi"] if short_best else None
                row[f"{p}_{c}_best_short_avg_move"] = _js(short_best["avg_short_move"]) if short_best else None
                if enable_advanced:
                    adv_long_best = best_adv[p][c]["long"][0] if best_adv[p][c]["long"] else None
                    adv_short_best = best_adv[p][c]["short"][0] if best_adv[p][c]["short"] else None
                    row[f"adv_{p}_{c}_best_long_rate"]     = _js(adv_long_best["rate"]) if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_total"]    = int(adv_long_best["total"]) if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_lo"]       = adv_long_best["lo"] if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_hi"]       = adv_long_best["hi"] if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_avg_move"] = _js(adv_long_best["avg_long_move"]) if adv_long_best else None
                    row[f"adv_{p}_{c}_best_short_rate"]     = _js(adv_short_best["rate"]) if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_total"]    = int(adv_short_best["total"]) if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_lo"]       = adv_short_best["lo"] if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_hi"]       = adv_short_best["hi"] if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_avg_move"] = _js(adv_short_best["avg_short_move"]) if adv_short_best else None
        row.update({"corr": _js(corr_s), "beta": _js(beta_s)})
        pd.DataFrame([row], columns=summary_cols).to_csv(output_summary_csv, mode="a", header=False, index=False)

        bp_row = {"ticker": cur_ticker, "bench": bench_seen, "best": {"standard": best_std}}
        if enable_advanced:
            bp_row["best"]["advanced"] = best_adv
        best_params_f.write(json.dumps(bp_row, ensure_ascii=False) + "\n")

        _reset_ticker()

    # ── chunk processor ───────────────────────────────────────────────────────
    def _process_chunk(chunk, ci):
        nonlocal cur_ticker, cur_day, bench_seen, static_set, corr_s, beta_s
        nonlocal day_entry, day_entry_dist, day_hour_entry

        req = {"ticker", "date", "dt"}
        if not req.issubset(chunk.columns):
            raise KeyError(f"Missing columns: {sorted(req - set(chunk.columns))}")

        if not assume_sorted:
            chunk["dt"] = pd.to_datetime(chunk["dt"], errors="coerce", utc=True)
            chunk.sort_values(["ticker", "date", "dt"], inplace=True)

        def _col(name):
            return chunk[name] if name in chunk.columns else pd.Series(np.nan, index=chunk.index)

        s_dt = pd.to_datetime(_col("dt"), errors="coerce", utc=True)
        ok   = s_dt.notna().to_numpy(copy=False)
        if not ok.any(): return

        s_dt2  = s_dt[ok]
        h_arr  = s_dt2.dt.hour.to_numpy(dtype="int16", copy=False)
        m_arr  = s_dt2.dt.minute.to_numpy(dtype="int16", copy=False)
        tk_arr = _col("ticker")[ok].to_numpy(copy=False)
        ds_arr = _col("date")[ok].to_numpy(copy=False)

        stock_arr  = pd.to_numeric(_col(STOCK_NUM_FIELD)[ok],  errors="coerce").to_numpy(dtype="float64", copy=False)
        bench_arr  = pd.to_numeric(_col(BENCH_NUM_FIELD)[ok],  errors="coerce").to_numpy(dtype="float64", copy=False)
        devsig_arr = pd.to_numeric(_col(DEVSIG_FIELD)[ok],     errors="coerce").to_numpy(dtype="float64", copy=False)

        bn_arr   = _col("bench")[ok].to_numpy(copy=False) if "bench" in chunk.columns else None
        corr_arr = _col("corr")[ok].to_numpy(copy=False)  if "corr"  in chunk.columns else None
        beta_arr = _col("beta")[ok].to_numpy(copy=False)  if "beta"  in chunk.columns else None

        for i in range(len(tk_arr)):
            tk = tk_arr[i]
            ds = ds_arr[i]
            hh = int(h_arr[i]); mm = int(m_arr[i])
            t_min = hh * 60 + mm
            spct = float(stock_arr[i])
            bpct = float(bench_arr[i])
            dsig = float(devsig_arr[i])

            # ticker boundary
            if cur_ticker is not None and tk != cur_ticker:
                _flush()
                cur_ticker = tk; cur_day = ds
                _reset_day()

            if cur_ticker is None:
                cur_ticker = tk; cur_day = ds

            # static fields (bench label / corr / beta) — captured once per ticker
            if bn_arr is not None and bench_seen is None:
                v = bn_arr[i]
                if pd.notna(v) and str(v).strip():
                    bench_seen = str(v)

            if not static_set and corr_arr is not None and beta_arr is not None:
                c, b = corr_arr[i], beta_arr[i]
                if pd.notna(c) and pd.notna(b):
                    corr_s, beta_s = float(c), float(b)
                    static_set = True

            # day boundary
            if ds != cur_day:
                _finalize_day()
                cur_day = ds
                _reset_day()

            # ── standard entry: nearest point to entry_hm (09:20), searched from
            # BOTH sides within [entry_window_from, entry_window_to] ──
            if entry_hm_min <= t_min <= entry_hm_max and _ok(spct):
                dist = abs(t_min - entry_hm_target)
                if day_entry_dist is None or dist < day_entry_dist:
                    day_entry = {
                        "stack": spct,
                        "devsig": dsig if _ok(dsig) else None,
                        "bench": bpct if _ok(bpct) else None,
                    }
                    day_entry_dist = dist

            # ── standard exits ──
            for c, xt in exit_hm.items():
                if (hh, mm) == xt and c not in day_exits and _ok(spct):
                    day_exits[c] = spct

            # ── advanced: hourly checkpoints (exact H:00) + their H:10/H:30 exits ──
            if enable_advanced:
                if mm == 0 and _ok(spct):
                    day_hour_entry[hh] = {
                        "stack": spct,
                        "devsig": dsig if _ok(dsig) else None,
                        "bench": bpct if _ok(bpct) else None,
                    }
                for c in CLASSES:
                    offset_min = advanced_offset_minutes.get(c)
                    if offset_min is None:
                        continue
                    target_min = hh * 60 + offset_min
                    if t_min == target_min and hh in day_hour_entry:
                        day_hour_exits.setdefault(hh, {})
                        if c not in day_hour_exits[hh] and _ok(spct):
                            day_hour_exits[hh][c] = spct

    # ── main read loop ────────────────────────────────────────────────────────
    t0 = time.time()
    total_rows = 0
    last_rows  = 0
    last_ts    = t0
    is_parquet = str(input_path).lower().endswith((".parquet", ".pq", ".parq"))

    print(f"START OpenDoor v2  file={input_path}  parquet={is_parquet}")
    print(f"  entry window={entry_window_from}..{entry_window_to} target={entry_hm}  exits={exit_hm}")
    print(f"  min_events={min_events_per_ticker}  advanced={enable_advanced}")

    try:
        if is_parquet and parquet_use_pyarrow:
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(input_path)
            wanted = ["ticker", "date", "dt", "bench", "corr", "beta",
                      STOCK_NUM_FIELD, BENCH_NUM_FIELD, DEVSIG_FIELD]
            cols = [c for c in wanted if c in pf.schema.names]

            for ci in range(pf.num_row_groups):
                chunk = pf.read_row_group(ci, columns=cols).to_pandas()
                _process_chunk(chunk, ci + 1)
                total_rows += len(chunk)
                if (ci + 1) % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[rg {ci+1:>4}/{pf.num_row_groups}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:.1f}s")
                    last_rows, last_ts = total_rows, now
                del chunk
                if (ci + 1) % 20 == 0: gc.collect()

        elif not is_parquet:
            for ci, chunk in enumerate(
                pd.read_csv(input_path, compression="infer", low_memory=False, chunksize=csv_chunksize), 1
            ):
                _process_chunk(chunk, ci)
                total_rows += len(chunk)
                if ci % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[chunk {ci}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:.1f}s")
                    last_rows, last_ts = total_rows, now
                del chunk
                if ci % 20 == 0: gc.collect()

        else:
            df = pd.read_parquet(input_path)
            step = 1_000_000
            for ci, start in enumerate(range(0, len(df), step), 1):
                _process_chunk(df.iloc[start:start + step], ci)
                total_rows += len(df.iloc[start:start + step])

        _flush()
        elapsed = time.time() - t0
        print(f"DONE rows={total_rows:,} elapsed={elapsed:.1f}s")
        print(f"  onefile     = {output_onefile_jsonl}")
        print(f"  summary     = {output_summary_csv}")
        print(f"  best_params = {output_best_params_jsonl}")

    finally:
        onefile_f.close()
        best_params_f.close()


In [4]:
from pathlib import Path
import os


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path   = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve()   if sig_env   else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ── runner ────────────────────────────────────────────────────────────────────

FINAL_PATH, OUT_DIR = _resolve_orion_paths("opendoor")

opendoor_stats_v2_exporter(
    input_path=str(FINAL_PATH),
    output_onefile_jsonl=str(OUT_DIR / "onefile.jsonl.gz"),
    output_best_params_jsonl=str(OUT_DIR / "best_params.jsonl.gz"),
    output_summary_csv=str(OUT_DIR / "summary.csv"),
    entry_hm=(9, 20),
    entry_window_from=(9, 0),
    entry_window_to=(9, 25),
    exit_hm={"10m": (9, 40), "30m": (10, 0)},
    stack_bin_min=-15.0, stack_bin_max=15.0, stack_bin_step=1.0,
    bench_bin_min=-15.0, bench_bin_max=15.0, bench_bin_step=1.0,
    devsig_bin_min=-5.0, devsig_bin_max=5.0, devsig_bin_step=0.3,
    best_min_rate=0.60, best_min_total=10, min_events_per_ticker=10,
    STOCK_NUM_FIELD="Stack%", BENCH_NUM_FIELD="Bench%", DEVSIG_FIELD="DevSig",
    enable_advanced=True, advanced_offset_minutes={"10m": 10, "30m": 30}, advanced_hours=None,
    assume_sorted=True,
)


START OpenDoor v2  file=C:\datum-api-examples-main\OriON\CRACEN\final.parquet  parquet=True
  entry window=(9, 0)..(9, 25) target=(9, 20)  exits={'10m': (9, 40), '30m': (10, 0)}
  min_events=10  advanced=True


[rg    5/7640] rows=50,143 speed=176,788/s elapsed=0.3s
[rg   10/7640] rows=97,861 speed=715,627/s elapsed=0.4s


[rg   15/7640] rows=206,165 speed=649,278/s elapsed=0.5s
[rg   20/7640] rows=237,528 speed=469,135/s elapsed=0.6s
[rg   25/7640] rows=304,354 speed=573,145/s elapsed=0.7s


[rg   30/7640] rows=345,771 speed=620,594/s elapsed=0.8s
[rg   35/7640] rows=429,036 speed=624,273/s elapsed=0.9s


[rg   40/7640] rows=477,725 speed=405,185/s elapsed=1.0s
[rg   45/7640] rows=522,563 speed=460,222/s elapsed=1.1s
[rg   50/7640] rows=585,428 speed=607,791/s elapsed=1.2s


...

[rg 7610/7640] rows=72,448,311 speed=321,105/s elapsed=354.7s
[rg 7615/7640] rows=72,481,901 speed=217,006/s elapsed=354.8s


[rg 7620/7640] rows=72,535,869 speed=254,293/s elapsed=355.0s


[rg 7625/7640] rows=72,606,909 speed=266,165/s elapsed=355.3s
[rg 7630/7640] rows=72,637,370 speed=364,945/s elapsed=355.4s


[rg 7635/7640] rows=72,684,491 speed=148,746/s elapsed=355.7s
[rg 7640/7640] rows=72,728,646 speed=245,908/s elapsed=355.9s


DONE rows=72,728,646 elapsed=355.9s
  onefile     = C:\datum-api-examples-main\OriON\signals\opendoor\onefile.jsonl.gz
  summary     = C:\datum-api-examples-main\OriON\signals\opendoor\summary.csv
  best_params = C:\datum-api-examples-main\OriON\signals\opendoor\best_params.jsonl.gz
